# 00. Маршрут: видео → posterior → latent → видео

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Цель: отделить вероятностную модель, реализацию кодека, политику памяти и качество весов. После MIT Lecture 4 начните здесь; потом последовательно 01–08.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT до этой lab:** Lecture 4 slides 3–19; notes §6.2 pp.46–52. Посмотрите исходный [Lab 3 Part 4](../../lecture_04/references/lab_three_original.ipynb), zero-based cells 55–69; здесь мы строим общую карту.


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='00_route_and_source_contracts';metrics={}
print('Pinned source:',source_hash())


## 1. Четыре разных объекта исследования

1. **Статья v6**: §4.1 описывает causal 3D VAE, факторы сжатия; §4.1.1 — обучение с perceptual/adversarial losses; §4.1.2 — перекрывающиеся тайлы и дополнительный finetuning с tiling.
2. **Приложенный `hunyuan_vae.py`**: адаптация из Kandinsky с выбором spatial tiles по CUDA free bytes. Он побайтово совпал с уже закреплённым K6 snapshot; повторной копии не нужно.
3. **Native tiny**: оригинальные encoder/decoder blocks с уменьшенными ширинами, случайными весами, без optional mid attention. Это проверка вычислений/форм и свойств операторов.
4. **Pretrained reconstruction**: требует настоящего codec checkpoint и отдельного аппаратного запуска. Он отложен; мы не выдаём tiny reconstruction за результат модели.

Исходные `tiled_*` методы исполняются из AST без изменения тела. Для posterior/output используются подписанные учебные dependency adapters. Для planner CUDA budget имитируется только в локальном namespace. Весь проект не патчит `torch` глобально.


In [ ]:
import hashlib
attached=Path('/Users/vladmac/Downloads/hunyuan_vae.py')
if attached.exists():
    assert hashlib.sha256(attached.read_bytes()).hexdigest()==source_hash()
    print('Attached file == pinned source')
manifest=json.loads((COURSE/'sources.json').read_text())
print(json.dumps(manifest['paper'],ensure_ascii=False,indent=2))
metrics['attached_source_sha256']=source_hash()


## 2. Геометрия и единицы

В статье число кадров записано как T+1. Здесь используем F, чтобы не путать frame index, duration и коэффициент сжатия:

$$
X\in\mathbb{R}^{B\times3\times F\times H\times W},\qquad
M=E_\phi(X)\in\mathbb{R}^{B\times32\times F_z\times H_z\times W_z}.
$$

32 канала — два набора по 16: mean и log variance. Это **не 32 latent channels**.

$$
F_z=\lfloor(F-1)/4\rfloor+1,\quad H_z=H/8,\quad W_z=W/8.
$$

Равенство для H/W здесь предполагает делимость на 8; реальная свёртка с нечётным размером требует ceil и проверки. Для обратимого размера выбираем F=4k+1 и H/W кратные 8. Decoder возвращает 4(F_z−1)+1; произвольный F без padding/crop может не восстановить длину.

$$
N_X=3FHW,\qquad N_Z=16F_zH_zW_z,\qquad R=N_X/N_Z.
$$

R — сжатие числа **скаляров**, не ratio размера checkpoint и не число DiT tokens. Token patchification — следующий компонент.


In [ ]:
F,H,W=121,480,864
fz=(F-1)//4+1
nx,nz=3*F*H*W,16*fz*(H//8)*(W//8)
metrics['example']={'BCTHW':[1,3,F,H,W],'latent':[1,16,fz,H//8,W//8],
 'scalar_compression':nx/nz,'input_fp32_MiB':nx*4/2**20,'latent_fp32_MiB':nz*4/2**20}
assert 4*(fz-1)+1==F
print(metrics['example'])


## 3. План накопления знаний

| Lab | Математика | Код / измерение | Что объяснить своими словами |
|---|---|---|---|
| 01 | ELBO, Gaussian KL, reparameterization | MC integral, gradient, small VAE | Почему sample и mode различаются |
| 02 | Conv3D, receptive field, normalization | Native conv, mask, future perturbation | Где кончается строгая причинность |
| 03 | Encoder/decoder, posterior API | Native tiny hooks, contract check | Где появляются 32 и 16 каналов |
| 04 | Partition of unity, halo | Native spatial stitch, border coverage | Почему blend не восстанавливает контекст |
| 05 | Temporal lattice, first frame | Native temporal stitch, frame counts | Зачем F=4k+1 и почему этого мало для tile loop |
| 06 | Memory model, integer partition | Original planner budget sweep | Почему «optimal» не доказывает OOM safety |
| 07 | Controlled comparison | Three seeds, seams, CPU time, plots | Что числа позволяют заключить |
| 08 | Pretrained/manual validation | No-download runner + ComfyUI protocol | Как получить реальные memory/quality results |

Для связи с промышленной генерацией: [K5 VAE](../../kandinsky-5/notebooks/02_vae.ipynb), [K6 video codec](../../kandinsky-6/notebooks/03_video_vae.ipynb), [K6 latents/wrappers](../../kandinsky-6/notebooks/12_latents_conditioning_and_output.ipynb). VAE не решает Flow Matching ODE; он задаёт пространство, в котором DiT учит поле скоростей.


## Упражнения и критерий перехода

1. Посчитайте формы для F=1, 33, 122 и объясните потерю длины без padding.
2. Укажите, какие параметры обучаются при обучении VAE и какие frozen при DiT training.
3. Прочитайте Fig. 6 и сопоставьте каждый блок с исходником. Переходите дальше, когда можете отделить posterior parameters от latent sample.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
